# Pilot01 · Đo PatchCore end-to-end trên NVIDIA L4

**Người vận hành: Nguyễn Đình Huy. Người nhận log: Lê Quyết Tiến.** Chọn runtime L4 trong Colab và kiểm tra ngân sách compute của tài khoản trước khi kết nối. Notebook dùng ba bank đã huấn luyện, cùng 140 ảnh test_public và cùng phiên máy. Trước mỗi phép đo, điểm từng ảnh phải khớp CSV bàn giao. Lượt đã hoàn tất ghi vào thư mục mới dưới Drive.

Tạo shortcut tới [thư mục bàn giao](https://drive.google.com/drive/folders/171zEixi2Xb07Njhon9B_lsEORCbI6kz2) trong My Drive. Đặt tên shortcut là `pilot01-handover`, hoặc đổi ASSETS ở ô bên dưới. Thư mục phải có `data/mvtec_ad_2/vial/test_public` và `outputs/pilot01`. Ảnh RGB được giải mã trước khi bấm giờ. Kết quả gồm cả điểm và map 224² ở RAM. Notebook không tự đặt mua tài nguyên.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, csv, gc, datetime, platform
import torch
if not torch.cuda.is_available() or "L4" not in torch.cuda.get_device_name(0).upper().split():
    raise RuntimeError("Cần runtime NVIDIA L4 để ghi kết quả chính thức.")
from google.colab import drive
drive.mount("/content/drive")
ASSETS = Path("/content/drive/MyDrive/pilot01-handover")
PILOT = ASSETS / "outputs/pilot01"
DATA = ASSETS / "data/mvtec_ad_2"
assert (PILOT/"predictions.csv").is_file() and (DATA/"vial/test_public").is_dir()
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = Path("/content/drive/MyDrive/robust-memory-ad-results") / STAMP
print(torch.__version__, torch.cuda.get_device_name(0), OUTPUT)

In [ ]:
import tempfile
work = Path(tempfile.mkdtemp(prefix="huy-l4-",dir="/content"))
team, official = work/"team", work/"patchcore"
PROF = "15ab4dac972d60885aa6ec229d68727d52523bfe"
BASE = "6d7e8d8f0127838aae8f9c2dd3f2c85a87b94141"
UPSTREAM = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"
def checkout(url, ref, folder):
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    subprocess.run(["git","clone","-q","--no-checkout",url,str(folder)],env=env,check=True)
    subprocess.run(["git","-C",str(folder),"checkout","-q","--detach",ref],env=env,check=True)
    assert subprocess.check_output(["git","-C",str(folder),"rev-parse","HEAD"],text=True).strip()==ref
checkout("https://github.com/huyryan220806/robust-memory-ad.git",PROF,team)
checkout("https://github.com/amazon-science/patchcore-inspection.git",UPSTREAM,official)
os.chdir(team)
import importlib.metadata as metadata
# Keep Colab's matching torch and torchvision; the smaller dependencies follow the handover.
constraints = work/"constraints.txt"
constraints.write_text("\n".join(name+"=="+metadata.version(name) for name in ("torch","torchvision")))
subprocess.run([sys.executable,"-m","pip","install","-q","-c",str(constraints),"-r",str(team/"requirements-baseline-check.txt"),"psutil"],check=True)
sys.path[:0] = [str(team),str(official/"src")]

In [ ]:
from PIL import Image
import numpy as np
import patchcore.patchcore, patchcore.common
from scripts.baseline_checks import read_plain_params, read_reference_scores, verify_patchcore_checkout, sha256
from scripts.mvtec_ad2_dataset import MVTecAD2Dataset, DatasetSplit
from scripts.profile import run
verify_patchcore_checkout(official)
import random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
torch.backends.cudnn.benchmark=False
refs=read_reference_scores(PILOT/"predictions.csv")
assert sha256(PILOT/"predictions.csv")=="53ca1125b283dd90931ee05f64a7912ffabedeb0efc5d7248faa3e5ece711281"
rids=["pilot01-"+mode+"-s0" for mode in ("full","random","greedy")]
ids=sorted(i for rid,i in refs if rid==rids[0] and i.startswith("vial/test_public/"))
assert len(ids)==len(set(ids))==140
for rid in rids:
    assert set(ids)=={i for run_id,i in refs if run_id==rid and i.startswith("vial/test_public/")}
dataset=MVTecAD2Dataset(DATA,"vial",resize=256,imagesize=224,split=DatasetSplit.TEST)
sizes=list(csv.DictReader((team/"docs/evidence/handover-20261126/bank-sizes-and-cpu-reload.csv").open()))
bank_info={r["run_id"]:r for r in sizes}
images={i:Path("data/mvtec_ad_2")/(i+".png") for i in ids}
assert all((ASSETS/f).is_file() for f in images.values())
class Adapter:
    def load(self,cfg,base):
        folder=base/cfg["bank_dir"]
        self.device=torch.device(cfg["device"])
        self.model=patchcore.patchcore.PatchCore(self.device)
        read_plain_params(folder/"patchcore_params.pkl")
        self.model.load_from_path(str(folder),device=self.device,nn_method=patchcore.common.FaissNN(on_gpu=False,num_workers=4))
        self.model.eval()
        assert self.model.anomaly_scorer.nn_method.search_index.ntotal==cfg["num_patches"]
        ckpt = Path(torch.hub.get_dir())/"checkpoints/wide_resnet50_2-95faca4d.pth"
        assert sha256(ckpt)=="95faca4d11227dddf8633dbb5ff6c8a9003c1aa5b8945c73834b8007b10950b8"
    def decode(self,spec,base):
        with Image.open(base/spec["path"]) as im:
            return im.convert("RGB")
    def predict(self,im):
        scores,maps=self.model._predict(dataset.transform_img(im).unsqueeze(0).to(self.device))
        return scores,maps
    def image_score(self,result):
        scores,maps=result
        map0=np.asarray(maps[0])
        if map0.shape!=(224,224) or not np.isfinite(map0).all(): raise ValueError("Invalid map")
        return float(scores[0])
    def synchronize(self):
        pass  # FAISS CPU calls are synchronous; the profiler synchronizes CUDA.


In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=False)
(OUTPUT/"hardware.txt").write_text(subprocess.check_output(["lscpu"],text=True)+"\ntorch CUDA "+str(torch.version.cuda))
for rid in rids:
    bank=Path("outputs/pilot01/runs")/rid/"bank"
    idx=bank/"nnscorer_search_index.faiss"
    params=bank/"patchcore_params.pkl"
    # Verify SHA256 of content and baseline config before passing the bank to the model.
    assert sha256(ASSETS/idx)==bank_info[rid]["index_sha256"]
    assert subprocess.check_output(["git","-C",str(team),"rev-parse",PROF+":configs/pilot/"+rid+".json"])==subprocess.check_output(["git","-C",str(team),"rev-parse",BASE+":configs/pilot/"+rid+".json"])
    cfg=dict(kind="model_inference",run_id=rid,measurement_id=STAMP+"-"+rid,image_group="test_public_all",
             device="cuda:0", require_l4=True,baseline_commit=BASE,baseline_commit_verified=True,
             profiling_commit=PROF,patchcore_commit=UPSTREAM,seed=0,
             budget=int(bank_info[rid]["num_patches"]),num_patches=int(bank_info[rid]["num_patches"]),
             bank_dir=bank.as_posix(),bank_layout="FAISS flat L2 index + params",dtype="float32",
             resolution=[224,224],batch_size=1,query="FAISS CPU flat L2 1-NN; 4 threads",
             timing_boundary="decoded RGB in RAM to score and smoothed map in RAM",
             score_atol=1e-4,score_rtol=0,warmup=10,iterations=len(ids),
             images=[dict(image_id=i,path=images[i].as_posix(),expected_score=refs[(rid,i)]) for i in ids],
             artifacts=[dict(role="bank_index",path=idx.as_posix()),dict(role="metadata",path=params.as_posix())])
    run(cfg,Adapter(),OUTPUT/rid,ASSETS)
    gc.collect()
    torch.cuda.empty_cache()
    print(rid, (OUTPUT/rid/"efficiency.csv").read_text())
print("Gửi Tiến thư mục đầy đủ:", OUTPUT)

Mỗi run phải có `status.json` hoàn tất, `validation.json`, 140 hàng đo trong `timings.csv`, log warm-up, bảng `efficiency.csv` và metadata thiết bị. Khi phép kiểm score thất bại, giữ log lỗi, so ảnh/transform/FAISS và tạo measurement ID mới sau khi xử lý. Gửi cả thư mục cho Tiến để ghép với quality.csv; ba measurement vẫn chỉ là ba measurement dù bảng chất lượng có 216 hàng.